# Derive and save two-reference drift diagnostics

Consume the prepared monthly products from `0_run_drift_references.ipynb`, derive the complete May/November × five-variable × two-method diagnostic matrix, and save compact figure-ready NetCDF products plus a manifest. Existing complete products are reused: only missing regions or analyses are computed, while legacy single-region files receive a metadata-only schema upgrade when possible. This notebook performs no plotting; the `5a_*`, `5b_*`, and `5c_*` notebooks read its saved products. The calculation retains initialization year `Y`, averages only ensemble member `M`, intersects the valid reference cohort separately for every variable and initialization, and includes historical-spread-normalized attractor departures where `sigma_att` is available.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib
import sys

from IPython.display import display
import numpy as np
import pandas as pd
import xarray as xr
import esp_lab

REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from esp_lab.diagnostics.two_reference_drift import (
    area_weighted_mean, bootstrap_paired_mean_ci, compare_drift_skill_relationship,
    compare_initializations, compute_diagnostics, compute_early_drift_late_error,
    compute_regime_fraction, compute_spatial_drift_summary, regional_subset,
    run_pipeline as run_two_reference_pipeline,
)
from workflows.diagnostics import two_reference_drift as drift_workflow
drift_workflow = importlib.reload(drift_workflow)
apply_unit_transform = drift_workflow.apply_unit_transform
atomic_to_netcdf = drift_workflow.atomic_to_netcdf
discover_monthly_hindcast = drift_workflow.discover_monthly_hindcast
load_drift_references = drift_workflow.load_drift_references
select_initialization_years = drift_workflow.select_initialization_years

## 1. Analysis matrix and conventions

In [2]:
SOURCES = ('Reanalysis', 'JRA55_FOSIRL')
ANALYSIS_YEARS = (1980, 2017)
BASELINE_LEAD = 1
DISTANCE_TOLERANCE = 1.0e-6
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42
LEAD_WINDOWS = drift_workflow.DEFAULT_LEAD_WINDOWS
EARLY_LEADS = LEAD_WINDOWS['early_adjustment']
LATE_LEADS = LEAD_WINDOWS['extended_seasonal']
VARIABLE_COMPONENTS = {
    'TREFHT': 'atm',
    'SST': 'ocn',
    'PSL': 'atm',
    'PRECT': 'atm',
    'H2OSOI': 'lnd',
}
VARIABLE_ANALYSIS_YEARS = {'PRECT': (1980, 2015)}
VARIABLES_WITH_ATTRACTOR_SPREAD = {'TREFHT', 'SST', 'PSL', 'PRECT'}
ANALYSES = [
    {'variable': variable, 'component': component, 'init_month': init_month}
    for variable, component in VARIABLE_COMPONENTS.items()
    for init_month in (5, 11)
]
REGION_DEFINITIONS = {
    'Nino3.4': {'lon_bounds': (190, 240), 'lat_bounds': (-5, 5)},
    'North_Atlantic': {'lon_bounds': (280, 359.999), 'lat_bounds': (0, 60)},
    # Avoid (0, 360): regional_subset normalizes both endpoints to 0.
    'Global_land': {'lon_bounds': (0, 359.999), 'lat_bounds': (-90, 90)},
}
VARIABLE_REGIONS = {
    'TREFHT': ('Nino3.4', 'North_Atlantic'),
    'SST': ('Nino3.4', 'North_Atlantic'),
    'PSL': ('Nino3.4', 'North_Atlantic'),
    'PRECT': ('Nino3.4', 'North_Atlantic'),
    'H2OSOI': ('Global_land',),
}
OUTPUT_ROOT = Path('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REUSE_EXISTING_ANALYSIS_OUTPUTS = True
MONTH_NAMES = {5: 'May', 11: 'November'}
INIT_MONTHS = tuple(MONTH_NAMES)

## 2. Compute start-specific regional and spatial diagnostics

Regional series retain an explicit `region` dimension: Niño3.4 and North Atlantic for atmospheric/ocean variables, and global land for H2OSOI. Spatial fields remain start-specific through the drift calculation. Only afterward are they summarized across `Y` and explicit lead windows. Regime maps contain category frequencies, never averages of integer category codes. Where historical spread exists, `z_att = e_att / sigma_att` and non-positive spread is masked; H2OSOI retains the unnormalized attractor departure because its prepared reference intentionally has no `sigma_att`.

In [3]:
def _analysis_output_paths(variable, init_month):
    tag = f'{variable}_{init_month:02d}'
    paths = {}
    for source in SOURCES:
        paths[(source, 'regional')] = OUTPUT_ROOT / f'{source}_{tag}_regional.nc'
        paths[(source, 'spatial')] = OUTPUT_ROOT / f'{source}_{tag}_spatial_maps.nc'
    paths[('paired', 'regional')] = OUTPUT_ROOT / f'FOSIRL_minus_Reanalysis_{tag}_regional.nc'
    paths[('paired', 'spatial')] = OUTPUT_ROOT / f'FOSIRL_minus_Reanalysis_{tag}_spatial_maps.nc'
    return paths


def _load_dataset(path):
    with xr.open_dataset(path) as dataset:
        return dataset.load()


def _with_region_dimension(dataset, variable):
    if 'region' in dataset.dims:
        return dataset
    region_name = dataset.attrs.get('region')
    if region_name is None:
        region_name = (
            'Global_land' if variable == 'H2OSOI' else 'Nino3.4'
        )
    valid_time = dataset.coords.get('valid_time')
    expanded = dataset.expand_dims(region=[str(region_name)])
    if valid_time is not None:
        # valid_time describes Y/L and should not acquire a region axis.
        expanded = expanded.assign_coords(valid_time=valid_time)
    return expanded


def _regional_means(field, region_names):
    regional_fields = []
    for region_name in region_names:
        bounds = REGION_DEFINITIONS[region_name]
        regional_fields.append(
            area_weighted_mean(regional_subset(field, **bounds))
        )
    return xr.concat(regional_fields, dim='region').assign_coords(
        region=list(region_names)
    )


def _regional_cache_is_compatible(path, variable):
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        with xr.open_dataset(path) as dataset:
            return (
                'region' in dataset.dims
                and tuple(dataset.region.values.astype(str))
                == VARIABLE_REGIONS[variable]
            )
    except (OSError, ValueError, KeyError):
        return False


def _load_analysis_for_figures(variable, init_month):
    paths = _analysis_output_paths(variable, init_month)
    regional = {}
    skill = {}
    regime_fraction = {}
    spatial = {}
    drift_skill = {}
    regional_names = (
        'hindcast_mean', 'e_obs', 'e_att', 'delta_e_obs', 'delta_e_att',
        'delta_abs_e_obs', 'delta_abs_e_att', 'regime',
    )

    for source in SOURCES:
        saved = _with_region_dimension(
            _load_dataset(paths[(source, 'regional')]), variable
        )
        available_regional_names = [
            name for name in (*regional_names, 'z_att') if name in saved
        ]
        regional[source] = saved[available_regional_names]
        skill_names = [name for name in saved.data_vars if name.startswith('skill_')]
        skill[source] = saved[skill_names].rename(
            {name: name.removeprefix('skill_') for name in skill_names}
        )
        fraction_names = [name for name in saved.data_vars if name.startswith('fraction_regime_')]
        regime_fraction[source] = saved[fraction_names]
        spatial[source] = _load_dataset(paths[(source, 'spatial')])
        drift_skill[source] = compute_early_drift_late_error(
            regional[source].delta_abs_e_obs, regional[source].e_obs,
            early_leads=EARLY_LEADS, late_leads=LATE_LEADS,
        )

    paired_saved = _with_region_dimension(
        _load_dataset(paths[('paired', 'regional')]), variable
    )
    paired_names = [
        f'delta_{name}' for name in (*regional_names, 'z_att')
        if f'delta_{name}' in paired_saved
    ]
    paired = paired_saved[paired_names]
    paired_fraction_names = [
        name for name in paired_saved.data_vars
        if name.startswith('delta_fraction_regime_')
    ]
    paired_regime_fraction = paired_saved[paired_fraction_names]
    paired_ci = {}
    for name in paired.data_vars:
        statistics = {
            statistic: paired_saved[f'{name}_{statistic}']
            for statistic in ('estimate', 'ci_lower', 'ci_upper')
            if f'{name}_{statistic}' in paired_saved
        }
        if statistics:
            paired_ci[name] = xr.Dataset(statistics)
    paired_spatial = _load_dataset(paths[('paired', 'spatial')])
    paired_drift_skill = compare_drift_skill_relationship(
        drift_skill['JRA55_FOSIRL'], drift_skill['Reanalysis']
    )
    return {
        'regional': regional, 'skill': skill,
        'regime_fraction': regime_fraction, 'paired': paired,
        'paired_ci': paired_ci,
        'paired_regime_fraction': paired_regime_fraction,
        'drift_skill': drift_skill,
        'paired_drift_skill': paired_drift_skill, 'spatial': spatial,
        'paired_spatial': paired_spatial,
    }


required_analysis_products = [
    (analysis['variable'], product, path)
    for analysis in ANALYSES
    for (_, product), path in _analysis_output_paths(
        analysis['variable'], analysis['init_month']
    ).items()
]
required_analysis_paths = [path for _, _, path in required_analysis_products]

results = {}
missing_regions_by_analysis = {}
analyses_to_compute = []
updated_analysis_keys = set()
for analysis in ANALYSES:
    key = (analysis['variable'], analysis['init_month'])
    variable, init_month = key
    paths = _analysis_output_paths(variable, init_month)
    paths_complete = all(
        path.is_file() and path.stat().st_size > 0
        for path in paths.values()
    )
    if REUSE_EXISTING_ANALYSIS_OUTPUTS and paths_complete:
        results[key] = _load_analysis_for_figures(*key)
        region_coordinates = [
            tuple(results[key]['regional'][source].region.values.astype(str))
            for source in SOURCES
        ] + [tuple(results[key]['paired'].region.values.astype(str))]
        if len(set(region_coordinates)) != 1:
            raise ValueError(
                f'Cached regional products disagree for {variable}, '
                f'init={init_month:02d}: {region_coordinates}'
            )
        available_regions = set(region_coordinates[0])
        missing_regions = tuple(
            region for region in VARIABLE_REGIONS[variable]
            if region not in available_regions
        )
        regional_paths = [
            paths[(source, 'regional')] for source in SOURCES
        ] + [paths[('paired', 'regional')]]
        if not all(
            _regional_cache_is_compatible(path, variable)
            for path in regional_paths
        ):
            # Rewrite legacy single-region files with an explicit region axis.
            updated_analysis_keys.add(key)
    else:
        missing_regions = VARIABLE_REGIONS[variable]
    missing_regions_by_analysis[key] = missing_regions
    if missing_regions:
        analyses_to_compute.append(analysis)

print(
    f'Loaded {len(results)} complete existing analyses; '
    f'{len(analyses_to_compute)} require computation and '
    f'{len(updated_analysis_keys)} require a regional schema update.'
)


def _merge_regions(existing, additional, expected_regions):
    merged = xr.concat([existing, additional], dim='region')
    if merged.get_index('region').duplicated().any():
        raise ValueError('Region merge produced duplicate coordinates')
    return merged.sel(region=list(expected_regions))

for analysis in analyses_to_compute:
    variable = analysis['variable']
    component = analysis['component']
    init_month = analysis['init_month']
    analysis_years = VARIABLE_ANALYSIS_YEARS.get(variable, ANALYSIS_YEARS)
    require_attractor_spread = variable in VARIABLES_WITH_ATTRACTOR_SPREAD
    key = (variable, init_month)
    region_names = missing_regions_by_analysis[key]
    compute_spatial = key not in results
    regional_hindcasts = {}
    regional_obs = {}
    regional_att = {}
    regional_spread = {}
    spatial = {}
    valid_times = {}

    for source in SOURCES:
        job = {
            'source': source, 'component': component, 'variable': variable,
            'init_month': init_month, 'regrid': True,
        }
        hindcast_path = discover_monthly_hindcast(job)
        with xr.open_dataset(hindcast_path, chunks={}) as hindcast_ds, load_drift_references(
            source, init_month, variable, component=component, hindcast=None,
            require_attractor_spread=require_attractor_spread,
        ) as references:
            values = select_initialization_years(
                apply_unit_transform(hindcast_ds[variable], component, variable),
                analysis_years,
            )
            common_years = np.intersect1d(values.Y.values, references.Y.values)
            if common_years.size < 2:
                raise ValueError(
                    f'{variable} {source} init={init_month:02d} has fewer than two common starts'
                )
            values = values.sel(Y=common_years)
            references = references.sel(Y=common_years)
            np.testing.assert_array_equal(references.L, values.L)
            np.testing.assert_array_equal(references.valid_time, hindcast_ds.time.sel(Y=values.Y))
            valid_times[source] = references.valid_time.load()

            regional_hindcasts[source] = _regional_means(
                values, region_names
            ).load()
            for destination, field in (
                (regional_obs, references.X_obs),
                (regional_att, references.X_att),
            ):
                destination[source] = _regional_means(
                    field, region_names
                ).load()
            if require_attractor_spread:
                regional_spread[source] = _regional_means(
                    references.sigma_att, region_names
                ).load()

            if compute_spatial:
                spatial_product = compute_diagnostics(
                    values, references.X_obs, references.X_att,
                    baseline_lead=BASELINE_LEAD,
                    distance_tolerance=DISTANCE_TOLERANCE,
                )
                if require_attractor_spread:
                    spatial_product['z_att'] = (
                        spatial_product.e_att
                        / references.sigma_att.where(references.sigma_att > 0)
                    ).assign_attrs(
                        units='1',
                        diagnostic=(
                            'departure from E3SM-LE attractor normalized '
                            'by historical ensemble spread'
                        ),
                    )
                spatial[source] = compute_spatial_drift_summary(
                    spatial_product, LEAD_WINDOWS
                ).compute()

    pipeline = run_two_reference_pipeline(
        regional_hindcasts, regional_obs, regional_att,
        attractor_spreads=regional_spread if require_attractor_spread else None,
        baseline_lead=BASELINE_LEAD,
        distance_tolerance=DISTANCE_TOLERANCE,
    )
    regional_products = {
        source: xr.Dataset({
            name: pipeline[name][source]
            for name in (
                'hindcast_mean', 'e_obs', 'e_att', 'delta_e_obs', 'delta_e_att',
                'delta_abs_e_obs', 'delta_abs_e_att', 'regime', 'z_att',
            )
            if name in pipeline
        }).assign_coords(valid_time=valid_times[source])
        for source in SOURCES
    }
    regime_fraction = {
        source: compute_regime_fraction(
            regional_products[source].regime, spatial_dims=(), sample_dims=('Y',)
        )
        for source in SOURCES
    }
    paired = pipeline['paired'].copy()
    if require_attractor_spread:
        paired['delta_z_att'] = compare_initializations(
            regional_products['JRA55_FOSIRL'].z_att, regional_products['Reanalysis'].z_att
        )
    paired_ci = {
        name: bootstrap_paired_mean_ci(
            field, n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED
        )
        for name, field in paired.data_vars.items() if 'Y' in field.dims
    }
    paired_regime_fraction = compare_initializations(
        regime_fraction['JRA55_FOSIRL'], regime_fraction['Reanalysis']
    ).rename({name: f'delta_{name}' for name in regime_fraction['Reanalysis'].data_vars})
    drift_skill = {
        source: compute_early_drift_late_error(
            regional_products[source].delta_abs_e_obs, regional_products[source].e_obs,
            early_leads=EARLY_LEADS, late_leads=LATE_LEADS,
        )
        for source in SOURCES
    }
    paired_drift_skill = compare_drift_skill_relationship(
        drift_skill['JRA55_FOSIRL'], drift_skill['Reanalysis']
    )
    if compute_spatial:
        paired_spatial = compare_initializations(
            spatial['JRA55_FOSIRL'], spatial['Reanalysis']
        ).rename({
            name: f'delta_{name}'
            for name in spatial['Reanalysis'].data_vars
        })
    else:
        paired_spatial = results[key]['paired_spatial']
    additional_result = {
        'regional': regional_products, 'skill': pipeline['skill'],
        'regime_fraction': regime_fraction, 'paired': paired,
        'paired_ci': paired_ci, 'paired_regime_fraction': paired_regime_fraction,
        'drift_skill': drift_skill,
        'paired_drift_skill': paired_drift_skill, 'spatial': spatial,
        'paired_spatial': paired_spatial,
    }
    if key in results:
        existing = results[key]
        expected_regions = VARIABLE_REGIONS[variable]
        merged_regional = {
            source: _merge_regions(
                existing['regional'][source],
                additional_result['regional'][source],
                expected_regions,
            )
            for source in SOURCES
        }
        merged_skill = {
            source: _merge_regions(
                existing['skill'][source],
                additional_result['skill'][source],
                expected_regions,
            )
            for source in SOURCES
        }
        merged_fraction = {
            source: _merge_regions(
                existing['regime_fraction'][source],
                additional_result['regime_fraction'][source],
                expected_regions,
            )
            for source in SOURCES
        }
        merged_paired = _merge_regions(
            existing['paired'], additional_result['paired'],
            expected_regions,
        )
        merged_paired_fraction = _merge_regions(
            existing['paired_regime_fraction'],
            additional_result['paired_regime_fraction'],
            expected_regions,
        )
        merged_drift_skill = {
            source: compute_early_drift_late_error(
                merged_regional[source].delta_abs_e_obs,
                merged_regional[source].e_obs,
                early_leads=EARLY_LEADS, late_leads=LATE_LEADS,
            )
            for source in SOURCES
        }
        merged_paired_ci = {
            name: bootstrap_paired_mean_ci(
                field, n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED
            )
            for name, field in merged_paired.data_vars.items()
            if 'Y' in field.dims
        }
        results[key] = {
            'regional': merged_regional, 'skill': merged_skill,
            'regime_fraction': merged_fraction,
            'paired': merged_paired, 'paired_ci': merged_paired_ci,
            'paired_regime_fraction': merged_paired_fraction,
            'drift_skill': merged_drift_skill,
            'paired_drift_skill': compare_drift_skill_relationship(
                merged_drift_skill['JRA55_FOSIRL'],
                merged_drift_skill['Reanalysis'],
            ),
            'spatial': existing['spatial'],
            'paired_spatial': existing['paired_spatial'],
        }
    else:
        results[key] = additional_result
    updated_analysis_keys.add(key)
    print(
        f'Computed {variable} {MONTH_NAMES[init_month]}: '
        f'Y={regional_products[SOURCES[0]].sizes["Y"]}, '
        f'L={regional_products[SOURCES[0]].sizes["L"]}, '
        f'years={analysis_years[0]}-{analysis_years[1]}, '
        f'spread_normalized={require_attractor_spread}'
    )

Loaded 10 complete existing analyses; 8 require computation and 10 require a regional schema update.
Computed TREFHT May: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed TREFHT November: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed SST May: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed SST November: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed PSL May: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed PSL November: Y=38, L=24, years=1980-2017, spread_normalized=True
Computed PRECT May: Y=36, L=24, years=1980-2015, spread_normalized=True
Computed PRECT November: Y=36, L=24, years=1980-2015, spread_normalized=True


## 3. Export compact, reusable products

In [4]:
output_paths = []
summary_rows = []
results_to_write = {key: results[key] for key in updated_analysis_keys}
for (variable, init_month), result in results_to_write.items():
    tag = f'{variable}_{init_month:02d}'
    for source in SOURCES:
        regional = xr.merge(
            [result['regional'][source],
             result['skill'][source].rename({name: f'skill_{name}' for name in result['skill'][source].data_vars}),
             result['regime_fraction'][source]],
            compat='override', join='exact',
        )
        regional.attrs.update(
            source=source, variable=variable, initialization_month=init_month,
            regions=','.join(VARIABLE_REGIONS[variable]),
            region_definitions=str({
                name: REGION_DEFINITIONS[name]
                for name in VARIABLE_REGIONS[variable]
            }),
            analysis_years=str(VARIABLE_ANALYSIS_YEARS.get(variable, ANALYSIS_YEARS)),
            normalization=(
                'z_att=e_att/sigma_att'
                if variable in VARIABLES_WITH_ATTRACTOR_SPREAD else 'not available'
            ),
        )
        path = OUTPUT_ROOT / f'{source}_{tag}_regional.nc'
        atomic_to_netcdf(regional, path)
        output_paths.append(path)
        spatial_path = OUTPUT_ROOT / f'{source}_{tag}_spatial_maps.nc'
        if not spatial_path.is_file() or spatial_path.stat().st_size == 0:
            atomic_to_netcdf(result['spatial'][source], spatial_path)
        output_paths.append(spatial_path)
        for region_name in VARIABLE_REGIONS[variable]:
            summary_rows.append({
                'variable': variable, 'region': region_name,
                'init_month': init_month, 'source': source,
                'drift_skill_correlation': float(
                    result['drift_skill'][source].correlation.sel(
                        region=region_name
                    )
                ),
            })

    paired_export = result['paired'].copy()
    paired_export = xr.merge(
        [paired_export, result['paired_regime_fraction']], compat='override', join='exact'
    )
    for name, interval in result['paired_ci'].items():
        for statistic in interval.data_vars:
            paired_export[f'{name}_{statistic}'] = interval[statistic]
    paired_export.attrs.update(
        comparison='JRA55_FOSIRL minus Reanalysis',
        regions=','.join(VARIABLE_REGIONS[variable]),
    )
    paired_path = OUTPUT_ROOT / f'FOSIRL_minus_Reanalysis_{tag}_regional.nc'
    atomic_to_netcdf(paired_export, paired_path)
    output_paths.append(paired_path)
    spatial_path = OUTPUT_ROOT / f'FOSIRL_minus_Reanalysis_{tag}_spatial_maps.nc'
    if not spatial_path.is_file() or spatial_path.stat().st_size == 0:
        atomic_to_netcdf(result['paired_spatial'], spatial_path)
    output_paths.append(spatial_path)
    for region_name in VARIABLE_REGIONS[variable]:
        summary_rows.append({
            'variable': variable, 'region': region_name,
            'init_month': init_month, 'source': 'paired_difference',
            'drift_skill_correlation': float(
                result['paired_drift_skill'].correlation.sel(region=region_name)
            ),
        })

summary_rows = []
for (variable, init_month), result in results.items():
    for source in SOURCES:
        for region_name in VARIABLE_REGIONS[variable]:
            summary_rows.append({
                'variable': variable, 'region': region_name,
                'init_month': init_month, 'source': source,
                'drift_skill_correlation': float(
                    result['drift_skill'][source].correlation.sel(
                        region=region_name
                    )
                ),
            })
    for region_name in VARIABLE_REGIONS[variable]:
        summary_rows.append({
            'variable': variable, 'region': region_name,
            'init_month': init_month, 'source': 'paired_difference',
            'drift_skill_correlation': float(
                result['paired_drift_skill'].correlation.sel(
                    region=region_name
                )
            ),
        })
summary_path = OUTPUT_ROOT / 'two_reference_drift_skill_summary.csv'
if not updated_analysis_keys:
    output_paths = required_analysis_paths.copy()
    if summary_path.is_file():
        output_paths.append(summary_path)
    print('All requested regions already exist; no analysis files rewritten.')
else:
    pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
    output_paths.append(summary_path)
output_paths

[PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/Reanalysis_PRECT_05_regional.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/Reanalysis_PRECT_05_spatial_maps.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/JRA55_FOSIRL_PRECT_05_regional.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/JRA55_FOSIRL_PRECT_05_spatial_maps.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/FOSIRL_minus_Reanalysis_PRECT_05_regional.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/FOSIRL_minus_Reanalysis_PRECT_05_spatial_maps.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/Reanalysis_PRECT_11_regional.nc'),
 PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_ref

## 4. Save the figure-data manifest

The NetCDF products above are the authoritative figure data. The `5a_*`, `5b_*`, and `5c_*` notebooks plot them without opening raw hindcasts or reference archives.

In [5]:
figure_data_rows = []
for analysis in ANALYSES:
    variable = analysis['variable']
    init_month = analysis['init_month']
    for (source, product), path in _analysis_output_paths(variable, init_month).items():
        figure_data_rows.append({
            'variable': variable,
            'init_month': init_month,
            'source': source,
            'product': product,
            'path': str(path),
            'exists': path.is_file(),
            'size_bytes': path.stat().st_size if path.is_file() else 0,
        })
figure_data_manifest = pd.DataFrame(figure_data_rows)
figure_manifest_path = OUTPUT_ROOT / 'two_reference_drift_figure_data_manifest.csv'
figure_data_manifest.to_csv(figure_manifest_path, index=False)
missing_figure_products = figure_data_manifest.loc[
    ~figure_data_manifest['exists'] | figure_data_manifest['size_bytes'].eq(0)
]
if len(missing_figure_products):
    raise FileNotFoundError(
        f'{len(missing_figure_products)} figure-data products are missing; analysis is incomplete'
    )
output_paths.append(figure_manifest_path)
display(figure_data_manifest)
print(f'Figure data manifest: {figure_manifest_path}')
print('Plot independently by running jupyter/5a_refactor_drift_analysis.ipynb')

,variable,init_month,source,product,path,exists,size_bytes
0,TREFHT,5,Reanalysis,regional,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,232281
1,TREFHT,5,Reanalysis,spatial,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,13635249
2,TREFHT,5,JRA55_FOSIRL,regional,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,233035
3,TREFHT,5,JRA55_FOSIRL,spatial,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,13647094
4,TREFHT,5,paired,regional,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,369030
5,TREFHT,5,paired,spatial,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,13114543
6,TREFHT,11,Reanalysis,regional,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,232072
7,TREFHT,11,Reanalysis,spatial,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,13626030
8,TREFHT,11,JRA55_FOSIRL,regional,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,233121
9,TREFHT,11,JRA55_FOSIRL,spatial,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimod...,True,13608420


Figure data manifest: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/two_reference/two_reference_drift_figure_data_manifest.csv
Plot independently by running jupyter/5a_refactor_drift_analysis.ipynb


## 5. Validation / smoke tests

In [6]:
assert set(results) == {
    (variable, month) for variable in VARIABLE_COMPONENTS for month in INIT_MONTHS
}
for (variable, init_month), result in results.items():
    analysis_years = VARIABLE_ANALYSIS_YEARS.get(variable, ANALYSIS_YEARS)
    expected_starts = analysis_years[1] - analysis_years[0] + 1
    expected_regions = VARIABLE_REGIONS[variable]
    for source in SOURCES:
        product = result['regional'][source]
        assert product.hindcast_mean.dims == ('region', 'Y', 'L')
        assert tuple(product.region.values.astype(str)) == expected_regions
        assert product.sizes['Y'] == expected_starts
        if variable in VARIABLES_WITH_ATTRACTOR_SPREAD:
            assert product.z_att.attrs['units'] == '1'
        else:
            assert 'z_att' not in product
        for name in ('delta_e_obs', 'delta_e_att', 'delta_abs_e_obs', 'delta_abs_e_att'):
            np.testing.assert_allclose(product[name].sel(L=BASELINE_LEAD).fillna(0), 0, atol=1e-12)
        regimes = set(np.unique(product.regime.values[np.isfinite(product.regime.values)]).astype(int))
        assert regimes <= {0, 1, 2, 3, 4}
        fractions = result['regime_fraction'][source].to_array().sum('variable')
        np.testing.assert_allclose(fractions.where(fractions.notnull(), 1), 1, atol=1e-10)
        assert {'lat', 'lon'} <= set(result['spatial'][source].dims)
        has_spatial_z = any(
            name.startswith('z_att_') for name in result['spatial'][source].data_vars
        )
        assert has_spatial_z == (variable in VARIABLES_WITH_ATTRACTOR_SPREAD)
    assert set(result['paired_regime_fraction'].data_vars) == {f'delta_fraction_regime_{code}' for code in range(5)}
    np.testing.assert_array_equal(
        result['regional']['Reanalysis'].Y, result['regional']['JRA55_FOSIRL'].Y
    )
    assert result['paired'].attrs.get('comparison') == 'JRA55_FOSIRL minus Reanalysis'
print(f'Validated {len(results)} analyses and {len(output_paths)} figure-ready outputs.')

Validated 10 analyses and 62 figure-ready outputs.
